# 📊 개선 효과 검증 — 선박기기 매뉴얼 검색 PoC

**목적**: "기존 방식(키워드/수동 탐색) 대비 AI 검색의 개선 효과"를 재현 가능하게 정량 검증한다.
문제 정의서의 **성공 기준(① 30초 이내 출력 ② 출처·페이지 제공)** 도 함께 확인한다.

- **평가 대상**: `산소_농도_검출장치` 세트 (영문 매뉴얼 1종, 45페이지 → 258 청크)
- **실행 환경**: 로컬 CPU (Windows · Python 3.14)

> ⚠️ 코드 셀은 **재현용 발췌**로 실행 출력은 비워 두었고, 아래 표의 수치는 이 코드를
> 실제 실행해 얻은 **실측값**입니다(매뉴얼 원문은 보안상 미포함, 질의어·페이지번호·지표만 게시).

## 1. 평가 방법

- **질의셋**: 한글 10개 + 영어 5개 = 15개 (일반 도메인 표현으로 작성)
- **정답(gold)**: 각 질의의 정답 **페이지 번호**(매뉴얼 목차·내용 기준 라벨링)
- **지표 hit@5**: 정답 페이지가 **검색 상위 5개 페이지 안**에 포함되면 성공. *임계값을 적용하지 않고 순위로만* 판정(공정 비교).
- **비교 대상**
  - **키워드 검색** = 글자 일치(기존 `Ctrl+F`류 방식 재현)
  - **의미 검색(AI)** = 다국어 임베딩 `bge-m3` 코사인 유사도

In [ ]:
# 재현용 평가 코드 (실행: pip install -r requirements.txt 후 해당 세트 색인 필요)
import time, numpy as np, app

SET = "산소_농도_검출장치"
_, index_path, _, _ = app.set_paths(SET)
d = np.load(index_path, allow_pickle=True)
chunk_texts, chunk_labels, text_emb = d["chunk_texts"], d["chunk_labels"], d["text_embeddings"]
pages = np.array([int(str(l).split(" · ")[-1][1:]) for l in chunk_labels])  # 'p12' -> 12

# (질의, 언어, 정답 페이지 집합) — 일반 도메인 표현
QUERIES = [
    ("산소 분석기 사양","KO",{8}), ("센서 설치 방법","KO",{9,10}), ("전기 배선 연결","KO",{12}),
    ("센서 결선","KO",{13}), ("역세척 주기 설정","KO",{30}), ("센서 교정 절차","KO",{32,33,34}),
    ("스팬 가스 교정","KO",{34}), ("정기 점검 정비","KO",{35,36}), ("고장 진단 조치","KO",{37,38}),
    ("예비 부품 목록","KO",{39,40}), ("analyzer specifications","EN",{8}),
    ("sensor electrical connection","EN",{12,13}), ("span calibration gas","EN",{34}),
    ("trouble shooting","EN",{37,38}), ("spare parts list","EN",{39,40}),
]

def top5(scores):
    seen, out = set(), []
    for i in np.argsort(-scores):
        p = int(pages[i])
        if p not in seen:
            seen.add(p); out.append(p)
        if len(out) == 5: break
    return out

def keyword_top5(q):
    terms = app._terms(q)
    sc = np.array([sum(t in str(x).lower() for t in terms) for x in chunk_texts], float)
    return top5(sc) if sc.max() > 0 else []

def semantic_top5(q):
    qv = app.embed_texts_semantic([q], is_query=True)[0]
    return top5(text_emb @ qv)

for q, lang, gold in QUERIES:
    kw, se = keyword_top5(q), semantic_top5(q)
    print(lang, q, "kwHit", int(bool(set(kw)&gold)), "seHit", int(bool(set(se)&gold)))

## 2. 결과 ① 정확도 (hit@5) — 키워드 vs 의미검색  *(실측)*

| 질의 언어 | 키워드 검색 (기존) | 의미 검색 (AI) |
|---|---|---|
| 🇰🇷 한글 (10개) | **0 %** (0/10) | **90 %** (9/10) |
| 🇬🇧 영어 (5개) | 80 % (4/5) | 100 % (5/5) |
| **전체 (15개)** | **27 %** (4/15) | **93 %** (14/15) |

### 해석
- **한글 질의에서 기존 키워드 방식은 0 %** — 매뉴얼이 영문이라 한글 글자가 본문에 없어 **한 건도 못 찾는다.**
- **AI 의미검색은 한글 질의도 90 %** — 번역 없이 `산소 분석기 사양` → 영문 `Specifications` 페이지를 찾아낸다.
- 즉 **"한글로 영문 매뉴얼을 검색"이 기존엔 불가능했으나 AI로 가능**해졌다 (0 % → 90 %). 이것이 핵심 개선.
- 영어 질의도 80 % → 100 %로 향상(키워드는 표현이 다르면 놓치지만 의미검색은 포착).

> 한글 1건 실패는 `전기 배선 연결`(정답 p12)로, 상위에 인접 페이지 p11·p13(같은 "전기 연결" 절)이 잡힌 근접 실패였다.

## 3. 결과 ② 검색 속도 — 성공 기준(30초 이내)  *(실측)*

| 구분 | 소요 시간 | 30초 기준 |
|---|---|---|
| 콜드스타트 (최초 1회, AI 모델 로딩 포함) | **5.74 s** | ✅ 이내 |
| 의미검색 (워밍 후) 평균 | **0.081 s** | ✅ 이내 |
| 의미검색 (워밍 후) 최대 | 0.088 s | ✅ 이내 |
| end-to-end 검색 1회 (이미지+텍스트) | **2.72 s** | ✅ 이내 |

- 모델은 **최초 1회만 로딩**(5.74초)하고, 이후 검색은 **0.1초 미만**으로 즉시 응답한다.
- 사람이 목차부터 페이지를 뒤지거나 HDD를 손수 검색하던 수십 초~수 분과 비교해 **성공 기준(30초)을 크게 만족**한다.

## 4. 결과 ③ 출처·페이지 제공 — 성공 기준(정확성)  *(실측)*

검색 결과 카드는 항상 **`파일 · p페이지 | 유사도`** 형식으로 출처를 표시한다.
예: `센서 교정 절차` 검색 → 상위 결과 출처 페이지 **p37 · p34 · p20 …** (사용자가 원문 페이지로 즉시 검증 가능).

→ "결과에 출처와 페이지 번호를 제공하여 정확성을 확보한다"는 성공 기준 충족.

## 5. 성공 기준 체크리스트

| 성공 기준 | 목표 | 결과 | 충족 |
|---|---|---|---|
| 속도 | 결과 출력까지 30초 이내 | 워밍 후 0.08s · e2e 2.7s (콜드 5.7s) | ✅ |
| 정확성 | 출처·페이지 번호 제공 | 모든 결과에 `파일·페이지` 표기 | ✅ |
| (부가) 개선 | 기존 대비 검색 향상 | 한글 hit@5 0 %→90 %, 전체 27 %→93 % | ✅ |

## 6. 결론 및 한계

**결론** — 기존 키워드 방식으로는 **한글 질의 검색이 사실상 불가능(0 %)** 했으나, AI 의미검색으로 **90 %**까지
끌어올렸고(전체 27 %→93 %), **30초 이내 응답 + 출처·페이지 제공**이라는 성공 기준을 모두 만족했다.

**한계**
- 표본이 질의 15개 · 매뉴얼 1종으로 작다(경향 확인 수준).
- 정답 라벨은 페이지 단위이며, 근접 페이지 실패 1건이 있었다.
- **도면(이미지) 검색 정확도는 라벨링이 어려워 이번 정량 평가에서 제외**(텍스트 중심 세트).
- 다음 단계: 질의셋 확대, 도면 정답 라벨 구축, 로컬 LLM 연결로 **RAG(근거 기반 답변)** 확장.

### 재현 방법
```bash
pip install -r requirements.txt          # (윈도우+NVIDIA면 torch만 CUDA 빌드로 교체)
python app.py                            # 앱으로 '산소_농도_검출장치' 세트 색인
# 이후 위 2절 코드 셀 실행 → 동일 지표 재현
```